In [5]:
import pandas as pd
import os

# Base directories
base_dir = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata"
export_dir = os.path.join(base_dir, "csv_exports")
concepts_dir = os.path.join(base_dir, "csv_concepts_exports")
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_dir, exist_ok=True)

# --- Load source files ---
admissions = pd.read_csv(os.path.join(export_dir, "hosp_admissions.csv"))
patients = pd.read_csv(os.path.join(export_dir, "hosp_patients.csv"))  # for anchor_age

In [6]:
# --- Ensure proper datetime types ---
admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")
admissions["edregtime"] = pd.to_datetime(admissions["edregtime"], errors="coerce")
admissions["deathtime"] = pd.to_datetime(admissions["deathtime"], errors="coerce")

# Merge admissions (main table) with patients (for age) ---
encounter = admissions.merge(
    patients[["subject_id", "anchor_age"]],
    on="subject_id",
    how="left"
)

# Align columns to ENCOUNTER spec
encounter_final = pd.DataFrame({
    "csn": encounter["hadm_id"],
    "pat_id": encounter["subject_id"],
    "hospital_admission_date_time": encounter["admittime"],
    "hospital_discharge_date_time": encounter["dischtime"],
    "ed_presentation_time": encounter["edregtime"],
    "encounter_type": "IN",  # all MIMIC patients are inpatients
    "age": encounter["anchor_age"],  # use patients.anchor_age
    "discharge_to": encounter["discharge_location"],
    "pre_admit_location": encounter["admission_location"],
    "deathtime": encounter["deathtime"],
    "insurance": encounter["insurance"],
    "marital_status": encounter["marital_status"],
    "admission_type": encounter["admission_type"]
})

In [7]:

# Save to CSV
out_path = os.path.join(output_dir, "ENCOUNTER.csv")
encounter_final.to_csv(out_path, index=False)

print("✅ ENCOUNTER.csv generated:", out_path, "shape:", encounter_final.shape)
print(encounter_final.head())

✅ ENCOUNTER.csv generated: /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/ENCOUNTER.csv shape: (546028, 13)
        csn    pat_id hospital_admission_date_time  \
0  22595853  10000032          2180-05-06 22:23:00   
1  22841357  10000032          2180-06-26 18:27:00   
2  25742920  10000032          2180-08-05 23:44:00   
3  29079034  10000032          2180-07-23 12:35:00   
4  25022803  10000068          2160-03-03 23:16:00   

  hospital_discharge_date_time ed_presentation_time encounter_type  age  \
0          2180-05-07 17:15:00  2180-05-06 19:17:00             IN   52   
1          2180-06-27 18:49:00  2180-06-26 15:54:00             IN   52   
2          2180-08-07 17:50:00  2180-08-05 20:58:00             IN   52   
3          2180-07-25 17:55:00  2180-07-23 05:54:00             IN   52   
4          2160-03-04 06:26:00  2160-03-03 21:55:00             IN   19   

  discharge_to      pre_admit_location deathtime insurance marital_status  \
0         HOME  TRANSFE